Executar esses comandos no terminal antes de rodar os códigos:
* python -m venv venv
* .\venv\Scripts\Activate.ps1 
* pip install pandas numpy scikit-learn ipykernel  
* pip install tabulate

# Preparação dos Dados (Pré-processamento) 

In [ ]:
# ==========================================
# ETAPA 1: PREPARAÇÃO DO AMBIENTE E CARGA
# ==========================================

# Importamos as bibliotecas principais:
# Pandas: Essencial para manipulação e análise de dados em tabelas (DataFrames).
# OS: Usada para navegar e verificar pastas e arquivos no sistema operacional.
import pandas as pd
import numpy as np
import os

# Lista os arquivos dentro da pasta 'ml-1m' para confirmar que o dataset está acessível
print(os.listdir("ml-1m"))

# Importamos recursos específicos do Scikit-Learn:
# train_test_split: Para separar os dados em conjuntos de treino e teste.
# MinMaxScaler: Para normalizar as notas (rating) em uma escala comum (0 a 1).
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# --- VERIFICAÇÃO DE SEGURANÇA ---x
# Antes de carregar, checamos se a pasta com o dataset (MovieLens 1M) está visível.
# Isso garante que o caminho relativo ('ml-1m/') está correto em relação ao script.
if os.path.exists("ml-1m"):
    print("Pasta de dados encontrada!")
    print("Arquivos disponíveis:", os.listdir("ml-1m"))
else:
    print("ERRO: A pasta 'ml-1m' não foi encontrada no local atual.")

# --- CARREGAMENTO DOS DADOS ---
# Lemos o arquivo 'ratings.dat'.
# Definimos nomes manuais para as colunas, pois o arquivo original não possui cabeçalho.
df = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

# Exibe as 5 primeiras linhas para validar se a leitura foi feita corretamente
df.head()

['movies.dat', 'ratings.dat', 'README', 'users.dat']
Pasta de dados encontrada!
Arquivos disponíveis: ['movies.dat', 'ratings.dat', 'README', 'users.dat']


,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [219]:
# =========================================================
# ETAPA 2: EXPLORAÇÃO E AUDITORIA DE QUALIDADE
# =========================================================

# Ajustamos a configuração do Pandas para que números grandes e decimais
# sejam exibidos de forma legível, evitando a notação científica (ex: e+06).
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("--- INFORMAÇÕES TÉCNICAS ---")
# O .info() nos dá o "raio-X" do DataFrame:
# 1. Total de entradas (1.000.209)
# 2. Se as colunas são reconhecidas como números (int64/float64)
# 3. O uso de memória do sistema
print(f"[!] O dataset possui 1.000.209 linhas, 4 colunas, não existem valores nulos (non-null) e todos os dados são numéricos (int64)\n")
df.info()

print(f"\n--- LIMPEZA DE DADOS ---")
# Verificamos se existem "buracos" no dataset. 
# Esperamos 0 em todas as colunas, o que significa que o dataset está completo.

print(f"Valores ausentes:\n{df.isnull().sum()}")
# Verificamos se existem linhas idênticas repetidas.
# Dados duplicados podem "viciar" a IA (Overfitting), por isso o ideal é que o resultado seja 0.
print(f"\nRegistros duplicados: {df.duplicated().sum()}")
# Se ambos forem zero, o dataset é considerado "saudável" para o treinamento.

--- INFORMAÇÕES TÉCNICAS ---
[!] O dataset possui 1.000.209 linhas, 4 colunas, não existem valores nulos (non-null) e todos os dados são numéricos (int64)

<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   userId     1000209 non-null  int64
 1   movieId    1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB

--- LIMPEZA DE DADOS ---
Valores ausentes:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Registros duplicados: 0


In [220]:
# =========================================================
# ETAPA 3: NORMALIZAÇÃO E VALIDAÇÃO ESTATÍSTICA
# =========================================================

# Inicializamos o MinMaxScaler:
# Este recurso reescala os dados para que o menor valor vire 0 e o maior vire 1.    
scaler = MinMaxScaler()

# Aplicamos a normalização na coluna 'rating' (notas de 1 a 5).
# Criamos a nova coluna 'rating_normalizado' para manter o dado original intacto.
df['rating_normalizado'] = scaler.fit_transform(df[['rating']])

# Exibe as primeiras linhas para conferir a nova coluna lado a lado com a original
df.head()

print(f"[!] As notas originais vão de: 1 até 5. A normalização transforma para: 0 até 1\n")

print(f"\n--- RESUMO DA ESTATÍSTICA ---")
# Exibe o volume total formatado com pontos (padrão brasileiro)
print(f"Total de avaliações: {df.shape[0]:,}".replace(',', '.'))
# Mostra a tendência central das notas dos usuários
print(f"Média das notas originais: {df['rating'].mean():.2f}")
# Prova real de que a normalização atingiu os limites de 0 e 1
print(f"Variação após a normalização: {df['rating_normalizado'].min():.0f} até {df['rating_normalizado'].max():.0f}")

print(f"\nNota: A normalização (0 a 1) foi concluída com sucesso, permitindo que o modelo trate as notas de forma equilibrada.")
print("não dê peso excessivo a notas altas, tratando a variação de forma equilibrada.")

print(f"\n--- ESTATÍSTICA DESCRITIVA ---")
# O comando .describe() gera métricas como desvio padrão e quartis.
# Usamos o .T (transpor) para inverter a tabela, facilitando a leitura das colunas.
df.describe().T

[!] As notas originais vão de: 1 até 5. A normalização transforma para: 0 até 1


--- RESUMO DA ESTATÍSTICA ---
Total de avaliações: 1.000.209
Média das notas originais: 3.58
Variação após a normalização: 0 até 1

Nota: A normalização (0 a 1) foi concluída com sucesso, permitindo que o modelo trate as notas de forma equilibrada.
não dê peso excessivo a notas altas, tratando a variação de forma equilibrada.

--- ESTATÍSTICA DESCRITIVA ---


,count,mean,std,min,25%,50%,75%,max
userId,1000209.00,3024.51,1728.41,1.00,1506.00,3070.00,4476.00,6040.00
movieId,1000209.00,1865.54,1096.04,1.00,1030.00,1835.00,2770.00,3952.00
rating,1000209.00,3.58,1.12,1.00,3.00,4.00,4.00,5.00
timestamp,1000209.00,972243695.40,12152558.94,956703932.00,965302637.00,973018006.00,975220939.00,1046454590.00
rating_normalizado,1000209.00,0.65,0.28,0.00,0.50,0.75,0.75,1.00


In [221]:
# =========================================================
# ETAPA 4: DIVISÃO DE TREINO E TESTE
# =========================================================

# Definimos as variáveis do modelo:
# X (Entradas/Features): Usamos as IDs de usuário e filme para a recomendação.
# y (Alvo/Target): É o valor que queremos prever (a nota normalizada).
x = df[['userId', 'movieId']]
y = df['rating_normalizado']

# Realizamos o "Split" dos dados:
# test_size=0.2: Reservamos 20% do dataset para testar a IA com dados inéditos.
# random_state=42: Garante que a divisão seja sempre a mesma ao rodar o código, 
# facilitando a comparação de resultados entre a equipe.
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# Exibe as dimensões dos conjuntos criados
# Espera-se ~800k linhas para treino e ~200k para teste.
print("Treino:", x_train.shape)
print("Teste:", x_test.shape)

# Nota de Apresentação:
# X → Dados de entrada (Quem avaliou e Qual filme)
# y → O que queremos prever (A nota)
# 80% treino e 20% teste

Treino: (800167, 2)
Teste: (200042, 2)


# Refinamento do Modelo (SVD via Sckikit-learn)

In [222]:
# =========================================================
# ETAPA 1: TREINAMENTO E REFINAMENTO DO MODELO SVD
# =========================================================
from sklearn.decomposition import TruncatedSVD
import time

print("--- INICIANDO TREINAMENTO DO MODELO ---")
start_time = time.time()

# 1. CRIANDO A MATRIZ DE RECOMENDAÇÃO
# fillna(0) é necessário pois o SVD não aceita valores nulos (NaN)
user_movie_matrix = df.pivot(index='userId', columns='movieId', values='rating_normalizado').fillna(0)

# 2. CONFIGURAÇÃO DO ALGORITMO
# --- AJUSTE DE HIPERPARÂMETROS ---
# Valores menores (ex: 20) perdiam muitos detalhes dos gostos apesar de ser mais rápido
# Valores maiores (ex: 300) aumentavam o custo computacional sem ganho real de precisão, perdendo capacidade de generalizar (Overfitting)
# Aumentar os componentes (n_components) melhora a precisão, mas exige mais CPU/RAM
n_componentes = 200
svd_model = TruncatedSVD(n_components=n_componentes, random_state=42)

# 3. EXECUÇÃO DO TREINAMENTO
matrix_svd = svd_model.fit_transform(user_movie_matrix)

# 4. RESULTADOS DO REFINAMENTO
duration = time.time() - start_time
variancia = svd_model.explained_variance_ratio_.sum()

print(f"✅ Modelo treinado em {duration:.2f} segundos.")
print(f"📊 Componentes Latentes: {n_componentes}")
print(f"📈 Variância Explicada Total: {variancia:.2%}")

--- INICIANDO TREINAMENTO DO MODELO ---
✅ Modelo treinado em 2.13 segundos.
📊 Componentes Latentes: 200
📈 Variância Explicada Total: 58.20%


In [227]:
# Carregando os nomes dos filmes (O "dicionário" do sistema)
df_movies = pd.read_csv("ml-1m/movies.dat", sep="::", engine="python", 
                        names=["movieId", "title", "genres"], encoding="latin-1")

print(f"Títulos de filmes carregados: {len(df_movies)} títulos encontrados.")

Títulos de filmes carregados: 3883 títulos encontrados.


In [224]:
# =========================================================
# ETAPA 2: DEMONSTRAÇÃO DE RESULTADOS (VALIDAÇÃO VISUAL)
# =========================================================
import numpy as np

def mostrar_recomendacoes(u_id, top_n):
    # Localiza o usuário na matriz treinada
    idx_u = user_movie_matrix.index.get_loc(u_id)
    vetor_u = matrix_svd[idx_u]

    # Identifica filmes que o usuário JÁ ASSISTIU
    # Usamos o 'df' original que contém todo o histórico
    ja_vistos = df[df['userId'] == u_id]['movieId'].unique()
    
    previsoes = []
    
    # Varre as colunas da matriz (que são os IDs dos filmes)
    for m_id in user_movie_matrix.columns:

        # Pula os filmes já vistos pelo usuário
        if m_id in ja_vistos:
            continue
        
        idx_col = user_movie_matrix.columns.get_loc(m_id)
        
        # O cálculo matemático (Produto Escalar)
        score = np.dot(vetor_u, svd_model.components_[:, idx_col])
        
        # Busca detalhes no dicionário de filmes
        info = df_movies[df_movies['movieId'] == m_id]
        if not info.empty:
            calc_estrelas = (score * 4) + 1
            
            # Aplica a trava: mínimo 1 e máximo 5
            estrelas_final = np.clip(calc_estrelas, 1, 5) # para que as previsões do SVD não ultrapassem 5 estrelas por conta dos pesos altos
            
            previsoes.append({
                "Filme": info['title'].values[0],
                "Previsão": round(estrelas_final, 2), # <-- Use aqui a variável com o clip!
                "Gênero": info['genres'].values[0]
            })

    # Ordena e reseta o índice para a tabela começar do 0 de forma limpa
    df_result = pd.DataFrame(previsoes).sort_values(by="Previsão", ascending=False)
    return df_result.head(top_n).reset_index(drop=True)

# --- EXECUÇÃO DO TESTE INTERATIVO ---
try:
    id_teste = int(input("Digite o ID do usuário (ex: 1): "))
    qtd_filmes = int(input("Digite a quantidade de filmes a ser recomendada: "))

    print(f"\n--- O QUE A IA RECOMENDA PARA O USUÁRIO {id_teste} ---")
    
    # Chamei a função passando os DOIS valores que você coletou
    display(mostrar_recomendacoes(id_teste, qtd_filmes))

except KeyError:
    print(f"❌ Erro: O ID {id_teste} não foi encontrado na base de dados.")
except ValueError:
    print("❌ Erro: Digite apenas números inteiros para o ID e para a quantidade.")


--- O QUE A IA RECOMENDA PARA O USUÁRIO 4156 ---


,Filme,Previsão,Gênero
0,Jackie Brown (1997),3.51,Crime|Drama
1,From Dusk Till Dawn (1996),3.32,Action|Comedy|Crime|Horror|Thriller
2,Falling Down (1993),3.27,Action|Drama
3,"Crow, The (1994)",3.26,Action|Romance|Thriller
4,Doctor Zhivago (1965),3.19,Drama|Romance|War
5,Live and Let Die (1973),3.13,Action
6,Outbreak (1995),3.11,Action|Drama|Thriller
7,Raging Bull (1980),3.10,Drama
8,Vertigo (1958),3.05,Mystery|Thriller
9,Rumble in the Bronx (1995),2.94,Action|Adventure|Crime


In [225]:
# Carregando o arquivo de notas (Histórico)
# Usamos o separador :: porque é o padrão do MovieLens 1M
df_ratings = pd.read_csv(
    "ml-1m/ratings.dat", 
    sep="::", 
    engine="python", 
    names=["userId", "movieId", "rating", "timestamp"],
    encoding="latin-1"
)

print(f"Histórico carregado: {len(df_ratings)} avaliações encontradas.")

Histórico carregado: 1000209 avaliações encontradas.


* HISTÓRICO DO USUÁRIO COMPARADO À PREVISÃO DO SVD

In [226]:
# Filtrando o histórico do Usuário
historico_usuario = df_ratings[df_ratings['userId'] == id_teste]

# Unindo com os nomes dos filmes para ficar legível
tabela_comparativa = pd.merge(historico_usuario, df_movies, on='movieId')

# Ordenando pelas maiores notas para ver o que ele mais gostou
tabela_comparativa = tabela_comparativa.sort_values(by='rating', ascending=False).reset_index(drop=True)

# Exibindo os top 10 filmes assistidos
print(f"Histórico de Filmes Assistidos pelo Usuário: {id_teste}")
display(tabela_comparativa[['title', 'genres', 'rating']].head(qtd_filmes))

Histórico de Filmes Assistidos pelo Usuário: 4156


,title,genres,rating
0,Yojimbo (1961),Comedy|Drama|Western,5
1,In the Line of Fire (1993),Action|Thriller,5
2,Air Force One (1997),Action|Thriller,5
3,"Pawnbroker, The (1965)",Drama,5
4,Kagemusha (1980),Drama|War,5
5,American Beauty (1999),Comedy|Drama,5
6,2001: A Space Odyssey (1968),Drama|Mystery|Sci-Fi|Thriller,5
7,Shanghai Noon (2000),Action,5
8,Seven (Se7en) (1995),Crime|Thriller,5
9,Midnight Cowboy (1969),Drama,5
